In [1]:
# Mrinal Pradhan
# Challenge 10
# Polynomial Fitting

import numpy as np
import matplotlib.pyplot as plt

def mse(yTrue, yPred):
    yTrue = np.asarray(yTrue, float)
    yPred = np.asarray(yPred, float)
    return np.mean((yTrue - yPred) ** 2)

def r2(yTrue, yPred):
    yTrue = np.asarray(yTrue, float)
    yPred = np.asarray(yPred, float)
    ssRes = np.sum((yTrue - yPred) ** 2)
    ssTot = np.sum((yTrue - np.mean(yTrue)) ** 2)
    return 1.0 - ssRes / ssTot if ssTot != 0 else 0.0

def fit_linear_lstsq(x, y):
    # y = m x + b
    A = np.vstack([x, np.ones(len(x))]).T
    m, b = np.linalg.lstsq(A, y, rcond=None)[0]
    return m, b

def predict_linear(x, m, b):
    return m * x + b

def fit_poly_polyfit(x, y, deg):
    # returns coefficients in descending powers (like np.poly1d)
    return np.polyfit(x, y, deg)

def predict_poly(x, coeffs):
    return np.polyval(coeffs, x)

def fit_poly_vandermonde_lstsq(x, y, deg):
    # Fit polynomial using Vandermonde matrix with least squares
    # V has columns [x^deg, x^(deg-1), ..., x^0]
    V = np.vander(x, N=deg + 1, increasing=False)
    coeffs = np.linalg.lstsq(V, y, rcond=None)[0]
    return coeffs

def train_test_split_indices(n, test_frac=0.3, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    idx = np.arange(n)
    rng.shuffle(idx)
    testSize = int(round(n * test_frac))
    testIdx = idx[:testSize]
    trainIdx = idx[testSize:]
    return trainIdx, testIdx

def print_fit_report(name, yTrue, yPred):
    print(f"{name:>22}  MSE={mse(yTrue, yPred):.6f}   R^2={r2(yTrue, yPred):.6f}")


# Sort X, preserve pairing

rng = np.random.default_rng(123)
n = 30
X = rng.random(n)
Y = rng.random(n)

order = np.argsort(X)
X = X[order]
Y = Y[order]

# Part A: Linear Least Squares vs cubic polynomial (polyfit)

print("Part A: Linear least squares vs cubic polyfit\n")
# Linear
mLin, bLin = fit_linear_lstsq(X, Y)
YhatLin = predict_linear(X, mLin, bLin)

# Cubic polyfit
degA = 3
cubicCoeffs = fit_poly_polyfit(X, Y, degA)
YhatCubic = predict_poly(X, cubicCoeffs)

print_fit_report("Linear (lstsq)", Y, YhatLin)
print_fit_report(" Cubic (polyfit)", Y, YhatCubic)

# Pick better by lower MSE, tie-break by higher R^2
linMse, cubicMse = mse(Y, YhatLin), mse(Y, YhatCubic)
linR2, cubicR2 = r2(Y, YhatLin), r2(Y, YhatCubic)

if (cubicMse < linMse) or (np.isclose(cubicMse, linMse) and cubicR2 > linR2):
    betterA = "Cubic polynomial"
else:
    betterA = "Linear"

print(f"\nBetter fit on this data = {betterA}")

# Plot Part A
xline = np.linspace(X.min(), X.max(), 400)
plt.figure()
plt.scatter(X, Y, s=30, label="Data")
plt.plot(xline, predict_linear(xline, mLin, bLin), label="Linear fit")
plt.plot(xline, predict_poly(xline, cubicCoeffs), label="Cubic polyfit")
plt.title("Linear vs Cubic fit")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.show()


# Part B: Increase degree until overfit, decrease until underfit

print("\nPart B: Linear Least Squares vs Polynomial fit\n")
train_idx, test_idx = train_test_split_indices(n, test_frac=0.3, rng=rng)
Xtr, Ytr = X[train_idx], Y[train_idx]
Xte, Yte = X[test_idx], Y[test_idx]

# Sort within train/test just for prettier plotting consistency
tr_order = np.argsort(Xtr)
te_order = np.argsort(Xte)
Xtr, Ytr = Xtr[tr_order], Ytr[tr_order]
Xte, Yte = Xte[te_order], Yte[te_order]

maxDeg = min(12, len(Xtr) - 1)  # cannot exceed train_points-1 for exact interpolation

results = []
best = None

for deg in range(1, maxDeg + 1):
    coeffs = fit_poly_polyfit(Xtr, Ytr, deg)
    pred_tr = predict_poly(Xtr, coeffs)
    pred_te = predict_poly(Xte, coeffs)

    row = {
        "deg": deg,
        "train_mse": mse(Ytr, pred_tr),
        "test_mse": mse(Yte, pred_te),
        "train_r2": r2(Ytr, pred_tr),
        "test_r2": r2(Yte, pred_te),
        "coeffs": coeffs
    }
    results.append(row)

    if best is None or row["test_mse"] < best["test_mse"]:
        best = row

print("deg | train_MSE  test_MSE   train_R2   test_R2")
for r in results:
    print(f"{r['deg']:>3} | {r['train_mse']:.6f}  {r['test_mse']:.6f}  {r['train_r2']:.6f}  {r['test_r2']:.6f}")

print(f"\nBest degree by lowest TEST MSE: degree {best['deg']}")

# Find a "blatant overfit" point:
overfit_deg = None
for r in results:
    if r["deg"] >= best["deg"] + 2 and r["train_mse"] < best["train_mse"] and r["test_mse"] > best["test_mse"] * 1.5:
        overfit_deg = r["deg"]
        break

# Find a "clear underfit" point:
underfit_deg = None
for r in results:
    if r["deg"] <= max(1, best["deg"] - 2) and r["test_mse"] > best["test_mse"] * 1.5:
        underfit_deg = r["deg"]
        break

if overfit_deg is None:
    overfit_deg = results[-1]["deg"]  # fallback: highest degree tested
if underfit_deg is None:
    underfit_deg = 1  # fallback: lowest degree

print(f"Overfit degree: {overfit_deg}")
print(f"Underfit degree: {underfit_deg}")

# Plot underfit, best, overfit on the same figure (train+test points)
def coeffs_for_deg(d):
    return next(r["coeffs"] for r in results if r["deg"] == d)

plt.figure()
plt.scatter(Xtr, Ytr, s=30, label="Train points")
plt.scatter(Xte, Yte, s=30, label="Test points")

xlineB = np.linspace(X.min(), X.max(), 500)
plt.plot(xlineB, predict_poly(xlineB, coeffs_for_deg(underfit_deg)), label=f"Underfit deg {underfit_deg}")
plt.plot(xlineB, predict_poly(xlineB, coeffs_for_deg(best["deg"])), label=f"Best deg {best['deg']}")
plt.plot(xlineB, predict_poly(xlineB, coeffs_for_deg(overfit_deg)), label=f"Overfit deg {overfit_deg}")

plt.title("Underfit vs Best vs Overfit")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.show()

# Part C: Vandermonde polynomial fit 
print("\nPart C: Vandermonde\n")

vandCoeffs = fit_poly_vandermonde_lstsq(X, Y, deg=3)
YhatVand = predict_poly(X, vandCoeffs)

print_fit_report("Cubic (polyfit)", Y, YhatCubic)
print_fit_report("Cubic (Vandermonde)", Y, YhatVand)

# Plot Part C
plt.figure()
plt.scatter(X, Y, s=30, label="Data")
plt.plot(xline, predict_poly(xline, cubicCoeffs), label="Cubic polyfit")
plt.plot(xline, predict_poly(xline, vandCoeffs), linestyle="--", label="Cubic Vandermonde")
plt.title("Part C: Cubic polyfit vs Vandermonde")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.show()

Part A: Linear least squares vs cubic polyfit

        Linear (lstsq)  MSE=0.080382   R^2=0.040147
       Cubic (polyfit)  MSE=0.079631   R^2=0.049115

Better fit on this data = Cubic polynomial


<Figure size 640x480 with 1 Axes>


Part B: Linear Least Squares vs Polynomial fit

deg | train_MSE  test_MSE   train_R2   test_R2
  1 | 0.084687  0.087744  0.000088  -0.253950
  2 | 0.084652  0.087233  0.000507  -0.246641
  3 | 0.084634  0.088093  0.000714  -0.258937
  4 | 0.083935  0.098179  0.008973  -0.403067
  5 | 0.080131  0.082611  0.053882  -0.180588
  6 | 0.076459  0.141997  0.097236  -1.029275
  7 | 0.068082  0.189423  0.196146  -1.707025
  8 | 0.044103  4.183464  0.479266  -58.785583
  9 | 0.036832  13.655873  0.565117  -194.155106
 10 | 0.022348  7.245989  0.736134  -102.551913
 11 | 0.022328  12.726843  0.736375  -180.878397
 12 | 0.021097  177.743212  0.750902  -2539.115465

Best degree by lowest TEST MSE: degree 5
Overfit degree: 7
Underfit degree: 1


<Figure size 640x480 with 1 Axes>


Part C: Vandermonde

       Cubic (polyfit)  MSE=0.079631   R^2=0.049115
   Cubic (Vandermonde)  MSE=0.079631   R^2=0.049115


<Figure size 640x480 with 1 Axes>